In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pad
import argparse
import matplotlib.pyplot as plt
import seaborn as sns
from prophet import Prophet

In [ ]:
file_path = "city_temperature.csv"
df_temperature = kagglehub.load_dataset(
            KaggleDatasetAdapter.PANDAS,
            "sudalairajkumar/daily-temperature-of-major-cities",
            file_path,)

file_path = "co2_conc.csv"
df_CO2_emission = kagglehub.load_dataset(
        KaggleDatasetAdapter.PANDAS,
        "arunavsutar/daily-atmosphere-carbon-dioxide-concentration",
        file_path,)

file_path = "sealevel.csv"
df_sea_level = kagglehub.load_dataset(
        KaggleDatasetAdapter.PANDAS,
        "kkhandekar/global-sea-level-1993-2021",
        file_path,)

In [ ]:
def EDA(self,name_df):

        if name_df == "sea_level":
            df_chosen=self.df_sea_level
        elif name_df == "temperature":
            df_chosen=self.df_temperature
        elif name_df == "CO2":
            df_chosen = self.df_CO2_emission
        elif name_df == "merge":
            df_chosen =  self.df_merged
        
        print(f'EDA on the data {name_df}:')
        print("First 5 records:\n", df_chosen.head(),"\n")
        print("Describe:\n",df_chosen.describe(),"\n")
        print("Shape\n", df_chosen.shape,"\n")
        print("Information\n")
        df_chosen.info() #print directly doesn't return anything
        print("Null values:\n", df_chosen.isnull().sum(),"\n")
        print("\n----------------------------------------------------------------------------------------------------------------------------------\n")
        input("Press enter to continue")
        print("\n\n\n\n\n")

In [ ]:
df_sea_level.drop(columns = ["TotalWeightedObservations","GMSL_noGIA","StdDevGMSL_noGIA","GMSL_GIA","StdDevGMSL_GIA","SmoothedGSML_GIA","SmoothedGSML_noGIA"])
df_sea_level = df_sea_level[(df_sea_level['Year'] >= 2013) & (df_sea_level['Year'] <= 2020)]
df_sea_level = df_sea_level.groupby(['Year']).mean().reset_index()

In [ ]:
df_temperature = df_temperature.drop(columns = ["Region","Country","State"])
df_temperature = df_temperature[(df_temperature['Year'] >= 2013) & (df_temperature['Year'] <= 2020)]
df_temperature  = df_temperature [df_temperature ['City'] == 'Paris']
df_temperature.rename(columns={'AvgTemperature': 'avg_city_temp'}, inplace=True)

In [ ]:
df_CO2_emission.drop(columns = ["Unnamed: 0","cycle"], inplace=True)
df_CO2_emission = df_CO2_emission[(df_CO2_emission['year'] >= 2013) & (df_CO2_emission['year'] <= 2020)]
df_CO2_emission.rename(columns={'year': 'Year'}, inplace=True)
df_CO2_emission.rename(columns={'month': 'Month'}, inplace=True)
df_CO2_emission.rename(columns={'day': 'Day'}, inplace=True)
df_CO2_emission.rename(columns={'trend': 'concentration_in_CO2'}, inplace=True)

In [ ]:
df_merged = df_temperature.merge(df_CO2_emission, on=['Year', 'Month', 'Day'], how='inner').merge(df_sea_level, on=['Year'], how='inner')

In [ ]:
df_paris = df_merged[df_merged['City'] == 'Paris']
corr_matrix = df_paris[["concentration_in_CO2","avg_city_temp","SmoothedGSML_GIA_sigremoved"]].corr()
plt.figure(figsize=(10, 7))
ax = sns.heatmap(corr_matrix, annot=True) # fmt="d" specifies the annotations' format as decimal integers (d stands for decimal)
plt.title("Matrice de corrélation")
plt.show()

In [ ]:
def predict_Prophet():

        df_training = df_merged[(df_merged["City"]== 'Paris')]
        df_training= df_training[df_training["avg_city_temp"] != -99]
        df_training['ds'] = pad.to_datetime(df_training[['Year', 'Month', 'Day']])
        df_training.rename(columns={'avg_city_temp': 'y'}, inplace=True) #need to rename it for the model
        df_training = df_training[['ds', 'y']]
        
        model = Prophet()
        model.fit(df_training)

        future = model.make_future_dataframe(periods=365 * 30)
        forecast = model.predict(future)

        print(forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail())
        
        fig = model.plot(forecast)
        plt.legend([
            'Prévision (yhat)', 
            'Incertitude basse (yhat_lower)', 
            'Incertitude haute (yhat_upper)', 
            'Observations'
        ], loc='upper left')
        plt.title("Prévision de la température sur 30 ans pour Paris")
        plt.xlabel("Date")
        plt.ylabel("Température")
        plt.show()

predict_Prophet()

In [ ]:
EDA("sea_level")
EDA("temperature")
EDA("CO2")